# Utilizando biblioteca de explicação de recomendações - RecSummarizer
---
Para gerar as recomendações será utilizada o *framework* **CaseRecommender** e para gerar explicações, será utilizada a bilbioteca **RecSummarizer**.

[CaseRecommender](https://github.com/caserec/CaseRecommender) é um *framework* com a implementação de técnicas populares de recomendação.

[RecSummarizer](https://github.com/luanssouza/recsummarizer) é uma biblioteca de explicações agnósticas ao modelo usando revisões de usuaŕios.


## Obtendo dependências

In [ ]:
!pip install git+https://github.com/luanssouza/recsummarizer -q
!pip install git+https://github.com/caserec/CaseRecommender -q
!pip install stanza -q

     |████████████████████████████████| 24.1 MB 1.7 MB/s 
     |████████████████████████████████| 85 kB 4.7 MB/s 
     |████████████████████████████████| 5.3 MB 26.4 MB/s 
     |████████████████████████████████| 1.3 MB 44.1 MB/s 
     |████████████████████████████████| 163 kB 46.1 MB/s 
     |████████████████████████████████| 7.6 MB 34.8 MB/s 
     |████████████████████████████████| 691 kB 7.4 MB/s 
     |████████████████████████████████| 216 kB 47.3 MB/s 


In [ ]:
# http://www.natcorp.ox.ac.uk/
!wget https://raw.githubusercontent.com/luanssouza/recsummarizer/main/resources/BNC_nouns.csv -q

In [ ]:
# http://jmcauley.ucsd.edu/data/amazon/links.html
!wget http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Musical_Instruments_5.json.gz -q
!wget http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_Musical_Instruments.json.gz -q
!wget http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/ratings_Musical_Instruments.csv -q

## Importando dependências 

In [ ]:
import json
import gzip
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
import stanza

# Baixando modelo para a língua inglesa
stanza.download('en', verbose=False)

# Criando pipeline de processamento de sentenças
nlp = stanza.Pipeline('en', processors='tokenize,mwt,pos,sentiment', verbose=False)

In [ ]:
# Importando módulos usados recsummarizer
from recsummarizer.item import StanzaItem
from recsummarizer.review import StanzaReview
from recsummarizer.corpus import CsvGeneralCorpus
from recsummarizer.extractor import epsilon_aspects_extraction
from recsummarizer.normalize.tf_idf_normalizer import TfIdfNormalizer
from recsummarizer.centroid.centroid import Centroid
from recsummarizer.persistence.stanza_persistence import StanzaPersistence
from recsummarizer.preprocess.stanza_preprocess import StanzaPreProcess
from recsummarizer.summarize.summarizer_baseline import SummarizerBaseline

In [ ]:
from recsummarizer.embedding.word2vec_embedding import Word2VecEmbedding
from recsummarizer.embedding.bert_embedding import BertEmbedding

# Obtendo modelo de línguagem pré-treinado Word2Vec
# Outros modelos: https://github.com/RaRe-Technologies/gensim-data
w2v_embedding = Word2VecEmbedding('glove-wiki-gigaword-50')

# Obtendo modelo de línguagem pré-treinado BERT
# Outros modelos: https://www.sbert.net/docs/pretrained_models.html
embedding = BertEmbedding('all-MiniLM-L6-v2') 

[==================================================] 100.0% 66.0/66.0MB downloaded


Downloading:   0%|          | 0.00/1.18k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/190 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Downloading:   0%|          | 0.00/466k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/350 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/232k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/349 [00:00<?, ?B/s]

## RecSummarizer

### Entendendo componentes

In [ ]:
raw_reviews = ["Barack Obama was born in Hawaii.", "The racer is really good."]

review = StanzaReview(raw_reviews[0], nlp)

print(review.raw_review)
print(review.nouns_occurrences)

Barack Obama was born in Hawaii.
Counter({'barack': 1, 'obama': 1, 'hawaii': 1})


In [ ]:
items = [
  {
    "id": 0, 
    "reviews": raw_reviews 
  }
]

general_corpus = CsvGeneralCorpus(pd.read_csv('./BNC_nouns.csv', index_col='noun'))

item = StanzaItem(0, raw_reviews, general_corpus, nlp)

item.kl_values()

item.aspects_score = epsilon_aspects_extraction(item.kl_nouns_values, -20)

item.top_k_aspects_evaluation(20)

item.sentence_filtering()

print(item.aspects_score)
print(item.filtered_sentences)
print(item.filtered_sentences_nn)

{'barack': inf, 'obama': 0.0, 'hawaii': -5.808142489980444, 'racer': -5.420534999272286}
['The racer is really good.']
[('The racer is really good.', <recsummarizer.sentence.sentence.Sentence object at 0x7ff2adc90d50>)]


### Pre-processamento

In [ ]:
# Creating normalizer instance
normalizer = TfIdfNormalizer()

# Creating centroid instance
centroid = Centroid(normalizer, 0.35)

# Creating persistence instance
persistence = StanzaPersistence('./data/', embedding, centroid)

# Creating a instance of PreProcess
preprocess = StanzaPreProcess(-20, 20)

# Preprocessing movies
preprocess.proprocess(items, persistence, general_corpus, nlp)

Item processed: 0


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


### Obtendo sumários

In [ ]:
print(SummarizerBaseline('./data/', 0.90, 5).summarize(0))

['The racer is really good.']


## CaseRecommender

In [ ]:
g = gzip.open("reviews_Musical_Instruments_5.json.gz", 'r')

reviews_data = []
for l in g:
  reviews_data.append(eval(l))

len(reviews_data)

10261

In [ ]:
reviews_data[4]

{'reviewerID': 'A94QU4C90B1AX',
 'asin': '1384719342',
 'reviewerName': 'SEAN MASLANKA',
 'helpful': [0, 0],
 'reviewText': "This pop filter is great. It looks and performs like a studio filter. If you're recording vocals this will eliminate the pops that gets recorded when you sing.",
 'overall': 5.0,
 'summary': 'No more pops when I record my vocals.',
 'unixReviewTime': 1392940800,
 'reviewTime': '02 21, 2014'}

In [ ]:
users_id = [r['reviewerID'] for r in reviews_data]
users_id = list(set(users_id))
len(users_id)

1429

In [ ]:
itens_id = [r['asin'] for r in reviews_data]
itens_id = list(set(itens_id))[:50]
len(itens_id)

50

In [ ]:
ratings = pd.read_csv('ratings_Musical_Instruments.csv', header=None)
ratings = ratings[ratings[1].isin(itens_id)]
ratings.columns = ['userId', 'movieId', 'rating', 'timestamp']
ratings

,userId,movieId,rating,timestamp
30305,A17PGDE8Z5NV8K,B0002CZVXM,5.0,1405641600
30306,A3B3UYKTFKZVO0,B0002CZVXM,5.0,1405555200
30307,A11BP81RMYOMGG,B0002CZVXM,5.0,1382400000
30308,A29D0KXIYLY6C7,B0002CZVXM,5.0,1357171200
30309,AJK15Q9JOEHRH,B0002CZVXM,5.0,1398038400
...,...,...,...,...
453763,A2Q7GW9O2SGS62,B00AHEWBM4,5.0,1360454400
453764,A1T2HY1BFBTFTV,B00AHEWBM4,5.0,1378857600
453765,A3DKSI3XTXGD44,B00AHEWBM4,5.0,1384560000
453766,A2B3MIAJLJHGFM,B00AHEWBM4,5.0,1377475200


In [ ]:
g = gzip.open("meta_Musical_Instruments.json.gz", 'r')

meta_data = []
for l in g:
  item = eval(l)
  if item['asin'] in itens_id:
    meta_data.append(item)

len(meta_data)

50

In [ ]:
map_users = {user: idx for idx, user in enumerate(ratings.userId.unique())}
map_items = {item: idx for idx, item in enumerate(ratings.movieId.unique())}
ratings['userId'] = ratings['userId'].map(map_users)
ratings['movieId'] = ratings['movieId'].map(map_items)

train, test = train_test_split(ratings, test_size=.2, random_state=2)
train.to_csv('train.dat', index=False, header=False, sep='\t')
test.to_csv('test.dat', index=False, header=False, sep='\t')

In [ ]:
map_title = {}
for meta in meta_data:
  if 'title' not in meta.keys():
    map_title[map_items[meta['asin']]] = None
    continue
  map_title[map_items[meta['asin']]] = meta['title']

In [ ]:
from caserec.recommenders.rating_prediction.matrixfactorization import MatrixFactorization

MatrixFactorization('train.dat', 'test.dat', 'rp_mf.dat', factors=3).compute()

[Case Recommender: Rating Prediction > Matrix Factorization]

train data:: 5048 users and 50 items (5213 interactions) | sparsity:: 97.93%
test data:: 1296 users and 50 items (1304 interactions) | sparsity:: 97.99%

training_time:: 0.236497 sec
prediction_time:: 0.010628 sec


Eval:: MAE: 0.694965 RMSE: 0.940086 


In [ ]:
predictions = pd.read_csv('rp_mf.dat', sep='\t', names=['user_id', 'movieId', 'rating'])
predictions['title'] = predictions.movieId.map(map_title)
predictions.head(3)

,user_id,movieId,rating,title
0,3,0,4.497039,"Dunlop Dual Design Straplok System, Silver"
1,4,0,4.484535,"Dunlop Dual Design Straplok System, Silver"
2,15,0,4.531532,"Dunlop Dual Design Straplok System, Silver"


### Pre-processamento

In [ ]:
items_reviews = {}

for r in reviews_data:
  if r['asin'] not in itens_id:
    continue
  if r['asin'] not in items_reviews.keys():
    items_reviews[r['asin']] = [r['reviewText']]
  elif len(items_reviews[r['asin']]) > 4:
    continue
  else:
    items_reviews[r['asin']].append(r['reviewText'])

In [ ]:
len(items_reviews['B0002CZVXM'])

5

In [ ]:
items = []

for k, v in map_items.items():
  items.append({'id': v, 'reviews': items_reviews[k]})

In [ ]:
general_corpus = CsvGeneralCorpus(pd.read_csv('./BNC_nouns.csv', index_col='noun'))

In [ ]:
# Creating normalizer instance
normalizer = TfIdfNormalizer()

# Creating centroid instance
centroid = Centroid(normalizer, 0.35)

# Creating persistence instance
persistence = StanzaPersistence('./data_amazon_5/', embedding, centroid)

# Creating a instance of PreProcess
preprocess = StanzaPreProcess(-20, 20)

# Preprocessing movies
preprocess.proprocess(items, persistence, general_corpus, nlp)

Item processed: 0


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


Item processed: 1
Item processed: 2
Item processed: 3
Item processed: 4
Item processed: 5
Item processed: 6
Item processed: 7
Item processed: 8
Item processed: 9
Item processed: 10
Item processed: 11
Item processed: 12
Item processed: 13
Item processed: 14
Item processed: 15
Item processed: 16
Item processed: 17
Item processed: 18
Item processed: 19
Item processed: 20
Item processed: 21
Item processed: 22
Item processed: 23
Item processed: 24
Item processed: 25
Item processed: 26
Item processed: 27
Item processed: 28
Item processed: 29
Item processed: 30
Item processed: 31
Item processed: 32
Item processed: 33
Item processed: 34
Item processed: 35
Item processed: 36
Item processed: 37
Item processed: 38
Item processed: 39
Item processed: 40
Item processed: 41
Item processed: 42
Item processed: 43
Item processed: 44
Item processed: 45
Item processed: 46
Item processed: 47
Item processed: 48
Item processed: 49


### Obtendo sumários

In [ ]:
print(" ".join(SummarizerBaseline('./data_amazon_5/', 0.90, 5).summarize(43)))

This is simply one of the best velcro tapes out there. This product makes your pedals SUPER SECURE.
